# Test Pipeline for AMFinder Training

In [ ]:
# Azure ML Libraries
from azure.ai.ml.entities import Environment
from azure.identity import InteractiveBrowserCredential
from azure.ai.ml import MLClient, Input, Output, command
from azure.ai.ml import dsl
from azure.ai.ml.dsl import pipeline
from azure.ai.ml import load_component
from azure.ai.ml.entities._assets.environment import BuildContext, Environment

## Pipeline configuration

In [ ]:
# True if the pipeline should be subitted to aml
run_pipeline = True
allow_preprep_caching = False
create_with_schedule = False

# Relevant variables
compute_target_name = "gpu-cluster-medium"
model_directory = ''
experiment_name = "252_test_run"
schedule_name = f"{experiment_name}-schedule"

# ML Client
azure_config = {
    "subscription_id": "redacted",
    "resource_group": "redacted",
    "workspace_name": "redacted",
    "storage_account": "redacted"
    }

subscription_id = azure_config["subscription_id"]
resource_group = azure_config["resource_group"]
workspace_name = azure_config["workspace_name"]
storage_account = azure_config["storage_account"]

credential=InteractiveBrowserCredential()

ml_client = MLClient(credential=credential,
                     subscription_id=subscription_id,
                     resource_group_name=resource_group,
                     workspace_name=workspace_name)

# Check if the compute compute resource is available
compute_resource = ml_client.compute.get(compute_target_name)

#deployment enviroment: - set default always to "qa"
#env = ml_client.environments.get(name="amf_aml_test_env", version="3")
#env = Environment()

#Potentially switch to definition on the fly by including the requirements_aml.txt right in the variable

# Select the enviroment for the python script steps
# pipeline_job_env = Environment(
#     image="mcr.microsoft.com/azureml/minimal-ubuntu22.04-py39-cuda11.8-gpu-inference:latest",
#     conda_file="./requirements_aml.yml"


## Default Paths

In [ ]:
# All registered data sources can be accessed through the ml client.
# It is also possible to add new data, like the temporary ones created below.
ml_ds_id = ml_client.datastores.get(storage_account).id
base_path = f"azureml:/{ml_ds_id}/paths/raw".replace("providers/Microsoft.MachineLearningServices/", "")

# Config base paths
small_subsample_dataset = base_path + r'/252_csv_Training/'
output_dir_path = base_path + r'/test_outdir/'

## Environment

In [ ]:
# Building the Environment in the UI allows for a specific Dockerfile to define commands, while using a conda yaml for further libraries.
pipeline_job_env = ml_client.environments.get(name="amf_env_2", version="24")
# The following allows building an Environment from PythonSDK remotely.
# pipeline_job_env = Environment(
#     build=BuildContext(path='./environment_files/',             #if BuildContext is used, a conda filepath is not allowed. If a yaml with conda dependencies is ought to be used, an image needs to be specified first. In this case we need apt-get libvips however.
#     dockerfile_path = 'Dockerfile'                              #path specifies the folder which shall be uploaded in the context, whereas Dockerfile is the specific file for the customised Docker image.
#     ),
#     name="amf_train_env",
#     version="20",
#     description="training_environment",
#    )

# pipeline_job_env = Environment(
#     image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
#     name="amf_train_env",
#     version="8",
#     conda_file="./environment_files/requirements_aml.yml",
#     description="training_environment",
#    )

#pipeline_job_env.validate()

## Pipeline step definition

In [ ]:
pipeline_step_AMFinder_252_test = command(
    name="amfinder252 testing",                              # Note to change this once switching to 252
    display_name="",
    description="",
    code="../amf_code",                                      # Thats the code uploaded as a "repo" to Azure
    inputs={
        "images": Input(type="uri_folder"),                  # specify uri_folder as we are working with a cloud folder in fact
    },
    outputs={
        "test_outputs": Output(type="uri_folder"),
    },
    command="""
            conda run -n requirements_aml python amf test \
            --use-csvs \
            --images ${{inputs.images}}\
            --outdir ${{outputs.test_outputs}} \
            """,
    environment=pipeline_job_env
)

In [ ]:
@dsl.pipeline(
    default_compute=compute_target_name,
    experiment_name=experiment_name,
    description="AMF252_Test") # Note to change this once switching to 252
def pipeline_amftest(images):
    job_test_amf_model = pipeline_step_AMFinder_252_test(
        images = images
    )

    return {
    }


In [ ]:
pipeline = pipeline_amftest(
    images = Input(type="uri_folder", path=small_subsample_dataset)
    )

pipeline_job = ml_client.jobs.create_or_update(
    pipeline,
    experiment_name=experiment_name,
    )
